In [352]:
import numpy as np
class Value:
    def __init__(self,data, _children=(), _op='', label=''):
        self._op=_op
        self.data=data
        self._backward=lambda:None
        self._prev=set(_children)
        self.label=label
        self.grad=0.0

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        
        out=Value(self.data+other.data, (self,other),'+')
        def _backward():
            self.grad+=1.0*out.grad
            other.grad+=1.0*out.grad
            
        out._backward=_backward
        return out
    
    def __radd__(self,other):
        return self+other

    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out=Value(self.data*other.data, (self,other),'*')
        def _backward():
            self.grad+=other.data*out.grad
            other.grad+=self.data*out.grad
        out._backward=_backward
        return out

    def __rmul__(self,other):
        return self*other
    
    def __neg__(self):
        return self*-1

    def __sub__(self,other):
        return self+(-other)

    def __rsub__(self,other):
        return (-self)+(other)

    def exp(self):
        x = self.data
        out = Value(np.exp(x),(self,),'exp')
        def _backward():
            self.grad+=out.data*out.grad
        out._backward=_backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self,other):
        return other *(self**-1)

    def __pow__(self,other):
        assert isinstance(other,(int,float))
        out = Value(self.data**other, (self,),f"**{other}")

        def _backward():
            self.grad+=((other)*(self.data**(other-1)))*out.grad

        out._backward=_backward
        return out

    def backward(self):
        topo=[]
        visited=set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad=1.0

        for n in reversed(topo):
            n._backward()

    def zero_grad(self):
        topo=[]
        visited=set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)

        build(self)
        self.grad=0.0

        for n in reversed(topo):
            n.grad=0.0

    def tanh(self):
        t=np.tanh(self.data)
        out = Value(np.tanh(self.data), (self,), 'tanh')
        def _backward():
            self.grad+= (1-t**2)*out.grad
        out._backward=_backward
        return out

    def log(self, base=np.e):
        out = Value(np.log(self.data)/np.log(base), (self,),f'log with base:{base}')
        def _backward():
            self.grad+=(1/self.data)*(np.log(base))*out.grad
        out._backward=_backward
        return out
    
    def relu(self):
        out = Value(self.data if self.data>0 else 0, (self,), "relu")
        def _backward():
            self.grad+=(0 if out.data==0 else 1)*out.grad
        out._backward=_backward
        return out

    def sigmoid(self):
        return 1 / (1 + (-self).exp())

In [353]:
import random
random.seed(42)
class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b=Value(random.uniform(-1,1))

    def parameters(self):
        return self.w +[self.b]

    def __call__(self,x):
        return (sum(a*b for a,b in zip(self.w,x))+self.b).tanh()

class Layer:
    def __init__(self,nin,nout):
        self.n=[Neuron(nin) for _ in range(nout)]

    def parameters(self):
        return [x for ne in self.n for x in ne.parameters()]

    def __call__(self,x):
        out =[ne(x) for ne in self.n]
        return out

class MLP:
    def __init__(self,nin,nouts):
        sz=[nin]+nouts
        self.layers=[Layer(sz[i],sz[i+1]) for i in range(len(nouts))]

    def parameters(self):
        return [x for l in self.layers for x in l.parameters()]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x)==1 else out